<a href="https://colab.research.google.com/github/ramailh02/UTS_BIG_DATA_RAMA/blob/main/UTS_BIG-DATA_RAMA-ILHAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install google-play-scraper

In [ ]:
from google_play_scraper import reviews, Sort
import csv

result, _ = reviews(
    'com.dto.sehatindonesiaku',
    lang='id',
    country='id',
    sort=Sort.NEWEST,
    count=100,
    filter_score_with=None
)

filename = 'ulasan_google_play.csv'


with open(filename, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['userName', 'score', 'at', 'content'])
    writer.writeheader()
    for review in result:

        writer.writerow({
            'userName': review['userName'],
            'score': review['score'],
            'at': review['at'],
            'content': review['content']
        })

print(f"Berhasil menyimpan {len(result)} ulasan ke '{filename}'")

In [ ]:
pip install transformers pandas matplotlib seaborn

Next, I'll load the pre-trained `w11wo/indonesian-roberta-base-prdect-id` model and its tokenizer from the `transformers` library. This model is designed for sequence classification, which is suitable for sentiment analysis.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import pandas as pd

tokenizer = AutoTokenizer.from_pretrained("w11wo/indonesian-roberta-base-prdect-id")
model = AutoModelForSequenceClassification.from_pretrained("w11wo/indonesian-roberta-base-prdect-id")

# The model's config often stores id2label mapping. Let's try to retrieve it.
if hasattr(model.config, 'id2label'):
    labels = [model.config.id2label[i] for i in range(len(model.config.id2label))]
else:
    # Fallback if id2label is not directly available, common for sentiment models
    labels = ['negative', 'neutral', 'positive'] # This is a common convention, adjust if model documentation says otherwise

print(f"Model labels: {labels}")

Now, I'll define a function that takes a review text, tokenizes it, feeds it to the model, and returns the predicted sentiment. The model will output logits, which I'll convert to probabilities and then map to one of the defined sentiment labels.

In [ ]:
def get_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=1)
    predicted_class_id = torch.argmax(probabilities, dim=1).item()

    # Map the predicted class ID to the corresponding label
    if 'labels' in globals() and predicted_class_id < len(labels):
        return labels[predicted_class_id]
    else:
        return f"Class_{predicted_class_id}" # Fallback if labels are not correctly mapped or defined

Next, I will load the `ulasan_sehat_Indonesiaku.csv` file into a pandas DataFrame and then apply the `get_sentiment` function to each review's content. This will add a new 'sentiment' column to your DataFrame.

In [7]:
# @title Default title text
df = pd.read_csv('/content/ulasan_sehat_Indonesiaku.csv') # Using the correct filename

df['sentiment'] = df['content'].apply(get_sentiment)

display(df.head())

,userName,score,at,content,sentiment
0,Sita Satria,3,2026-05-05 14:47:31,kenapa ya aplikasi ini apa eror kok saya ngent...,Fear
1,Melisa Kurniawati,3,2026-05-05 02:32:02,"coba adakan aplikasi untuk pc min, biar enak k...",Happy
2,Joni Mustopa (Oni),5,2026-05-04 01:05:57,mantap,Happy
3,Linda Wahyuni,2,2026-05-03 04:02:40,"ini aplikasi semakin di update semakin buruk, ...",Fear
4,Yan Mnubefor,3,2026-04-30 04:27:47,agak susah,Sadness


Finally, let's see the distribution of sentiments in your reviews and visualize it with a bar chart.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sentiment_counts = df['sentiment'].value_counts()
display(sentiment_counts)

plt.figure(figsize=(10, 6))
sns.barplot(x=sentiment_counts.index, y=sentiment_counts.values, hue=sentiment_counts.index, palette='viridis', legend=False)
plt.title('Distribution of Sentiments in Reviews')
plt.xlabel('Sentiment')
plt.ylabel('Number of Reviews')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()